# State-Dependent U.S. Equity Sector Rotation
## A Systematic Framework for Trend-Based Active Sector Allocation

# Block 5 — Benchmark Active Weights & Portfolio Engine

Block 5 does two jobs:

1. Construct the **conventional 12-month TSMOM active-sector benchmark** on the same inverse-volatility strategic base used by the state-dependent strategy.
2. Build a common execution-date portfolio engine so both active strategies are evaluated under the same Friday-signal → next-U.S.-session implementation rule.

## Benchmark TSMOM specification

For each sector:

- 12-month return > 0 → active multiplier `+0.25`
- 12-month return <= 0 → active multiplier `-0.25`

Raw sector weights are:

\[
w^{raw}_{i,t}=w^{Strategic}_{i,t}(1+A^{TSMOM}_{i,t})
\]

and are normalized cross-sectionally to sum to one.

## Execution convention

Target weights are calculated from the completed Friday-labelled signal week and become effective on the next U.S. trading session. Portfolio returns therefore use **execution-date target weights**, not same-week Friday weights.

This block creates comparable return series for:

- SPY passive benchmark
- strategic inverse-volatility portfolio
- inverse-volatility + 12m TSMOM tilts
- inverse-volatility + state-dependent tilts


In [1]:
# ============================================================
# BLOCK 5.1 — ENVIRONMENT, DRIVE & PRIOR MANIFESTS
# ============================================================

from pathlib import Path
from google.colab import drive
import json
from datetime import datetime, timezone

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation")

DIRS = {
    "root": PROJECT_ROOT,
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "outputs": PROJECT_ROOT / "outputs",
    "tables": PROJECT_ROOT / "outputs" / "tables",
    "manifests": PROJECT_ROOT / "manifests",
    "logs": PROJECT_ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

BLOCK1_MANIFEST = DIRS["manifests"] / "block_1_research_configuration.json"
BLOCK2_MANIFEST = DIRS["manifests"] / "block_2_universe_data_weights.json"
BLOCK3_MANIFEST = DIRS["manifests"] / "block_3_signal_engine.json"
BLOCK4_MANIFEST = DIRS["manifests"] / "block_4_state_active_weights.json"

for p in [BLOCK1_MANIFEST, BLOCK2_MANIFEST, BLOCK3_MANIFEST, BLOCK4_MANIFEST]:
    if not p.exists():
        raise FileNotFoundError(f"Required prior manifest not found: {p}")

with open(BLOCK1_MANIFEST, "r", encoding="utf-8") as f:
    block1 = json.load(f)

with open(BLOCK2_MANIFEST, "r", encoding="utf-8") as f:
    block2 = json.load(f)

with open(BLOCK3_MANIFEST, "r", encoding="utf-8") as f:
    block3 = json.load(f)

with open(BLOCK4_MANIFEST, "r", encoding="utf-8") as f:
    block4 = json.load(f)

CONFIG = block1["research_config"]
SECTOR_RECORDS = block1["sector_universe"]
SECTOR_TICKERS = [x["ticker"] for x in SECTOR_RECORDS]
BENCHMARK_TICKER = block1["benchmark_ticker"]

assert block2["strategic_weight_method"] == "INVERSE_VOLATILITY_52W"
assert block4["strategic_weight_method"] == "INVERSE_VOLATILITY_52W"
assert CONFIG["signal_frequency"] == "W-FRI"
assert CONFIG["rebalance_execution_rule"] == "NEXT_US_TRADING_SESSION_AFTER_SIGNAL"

print("Loaded and validated Blocks 1–4.")
print("Strategic allocation:", block2["strategic_weight_method"])
print("Canonical signal window:", block4["canonical_signal_start"], "to", block4["canonical_signal_end"])


Mounted at /content/drive
Loaded and validated Blocks 1–4.
Strategic allocation: INVERSE_VOLATILITY_52W
Canonical signal window: 2019-06-21 to 2026-08-21


In [2]:
# ============================================================
# BLOCK 5.2 — IMPORTS & LOAD REQUIRED DATA
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DAILY_CLOSE_PATH = Path(block2["saved_files"]["daily_adjusted_close"])
WEEKLY_TIMING_PATH = Path(block2["saved_files"]["weekly_signal_execution_calendar"])
STRATEGIC_WEIGHT_PATH = Path(block2["saved_files"]["strategic_weights"])
TSMOM_RETURN_PATH = Path(block3["saved_files"]["tsmom_returns"])
TSMOM_SIGN_PATH = Path(block3["saved_files"]["tsmom_signs"])
STATE_TARGET_PATH = Path(block4["saved_files"]["canonical_target_weights_wide"])

for p in [
    DAILY_CLOSE_PATH,
    WEEKLY_TIMING_PATH,
    STRATEGIC_WEIGHT_PATH,
    TSMOM_RETURN_PATH,
    TSMOM_SIGN_PATH,
    STATE_TARGET_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Required dataset not found: {p}")

daily_close = pd.read_parquet(DAILY_CLOSE_PATH)
weekly_timing = pd.read_parquet(WEEKLY_TIMING_PATH)
strategic_weights = pd.read_parquet(STRATEGIC_WEIGHT_PATH)
tsmom_returns = pd.read_parquet(TSMOM_RETURN_PATH)
tsmom_signs = pd.read_parquet(TSMOM_SIGN_PATH)
state_target_weights = pd.read_parquet(STATE_TARGET_PATH)

for obj in [daily_close, weekly_timing, strategic_weights, tsmom_returns, tsmom_signs, state_target_weights]:
    obj.index = pd.to_datetime(obj.index)

daily_close = daily_close.sort_index()
weekly_timing = weekly_timing.sort_index()
strategic_weights = strategic_weights.sort_index()
tsmom_returns = tsmom_returns.sort_index()
tsmom_signs = tsmom_signs.sort_index()
state_target_weights = state_target_weights.sort_index()

assert set(SECTOR_TICKERS).issubset(daily_close.columns)
assert BENCHMARK_TICKER in daily_close.columns
assert set(SECTOR_TICKERS).issubset(strategic_weights.columns)
assert set(SECTOR_TICKERS).issubset(tsmom_signs.columns)
assert set(SECTOR_TICKERS).issubset(state_target_weights.columns)

print("Loaded Block 5 inputs.")


Loaded Block 5 inputs.


In [3]:
# ============================================================
# BLOCK 5.3 — CANONICAL SIGNAL / EXECUTION WINDOW
# ============================================================

CANONICAL_SIGNAL_START = pd.Timestamp(block4["canonical_signal_start"])
CANONICAL_SIGNAL_END = pd.Timestamp(block4["canonical_signal_end"])

canonical_signal_weeks = weekly_timing.index[
    (weekly_timing.index >= CANONICAL_SIGNAL_START)
    & (weekly_timing.index <= CANONICAL_SIGNAL_END)
]

timing = weekly_timing.loc[canonical_signal_weeks].copy()
timing = timing.dropna(subset=["execution_date"]).copy()
timing["execution_date"] = pd.to_datetime(timing["execution_date"])

assert not timing["execution_date"].duplicated().any()
assert (timing["execution_date"] > timing["signal_observation_date"]).all()

print(f"Canonical executable signal weeks: {len(timing):,}")
print("First signal / execution:", timing.index.min().date(), "->", timing['execution_date'].iloc[0].date())
print("Last signal / execution:", timing.index.max().date(), "->", timing['execution_date'].iloc[-1].date())


Canonical executable signal weeks: 375
First signal / execution: 2019-06-21 -> 2019-06-24
Last signal / execution: 2026-08-21 -> 2026-08-24


In [4]:
# ============================================================
# BLOCK 5.4 — TSMOM ACTIVE MULTIPLIERS
# ============================================================

TSMOM_ACTIVE_MAGNITUDE = 0.25

canonical_tsmom_signs = (
    tsmom_signs
    .reindex(index=timing.index, columns=SECTOR_TICKERS)
)

if canonical_tsmom_signs.isna().any().any():
    bad = canonical_tsmom_signs.isna().stack()
    bad = bad[bad]
    raise ValueError(
        "Missing canonical TSMOM signs. First missing entries:\n"
        + str(bad.head(20))
    )

tsmom_active = canonical_tsmom_signs.replace({
    1.0: TSMOM_ACTIVE_MAGNITUDE,
    -1.0: -TSMOM_ACTIVE_MAGNITUDE,
    1: TSMOM_ACTIVE_MAGNITUDE,
    -1: -TSMOM_ACTIVE_MAGNITUDE,
})

valid_active_values = set(
    np.round(tsmom_active.stack().unique(), 10)
)
assert valid_active_values.issubset(
    {-TSMOM_ACTIVE_MAGNITUDE, TSMOM_ACTIVE_MAGNITUDE}
)

print("TSMOM active multiplier values:", sorted(valid_active_values))


TSMOM active multiplier values: [np.float64(-0.25), np.float64(0.25)]


In [5]:
# ============================================================
# BLOCK 5.5 — TSMOM TARGET WEIGHTS ON SAME STRATEGIC BASE
# ============================================================

strategic_canonical = (
    strategic_weights
    .reindex(index=timing.index, columns=SECTOR_TICKERS)
)

if strategic_canonical.isna().any().any():
    raise ValueError("Missing strategic weights in the canonical Block 5 window.")

tsmom_raw_weights = strategic_canonical * (1.0 + tsmom_active)

tsmom_target_weights = tsmom_raw_weights.div(
    tsmom_raw_weights.sum(axis=1),
    axis=0,
)

assert (tsmom_target_weights >= 0).all().all()
assert np.allclose(
    tsmom_target_weights.sum(axis=1).values,
    1.0,
    atol=1e-10,
)

print("TSMOM target weights constructed and normalized.")
display(tsmom_target_weights.head())


TSMOM target weights constructed and normalized.


Ticker,XLC,XLY,XLP,XLE,XLF,XLV,XLI,XLK,XLB,XLRE,XLU
Date,,,,,,,,,,,
2019-06-21,0.055675,0.089206,0.114466,0.045627,0.083022,0.088471,0.087986,0.085597,0.086223,0.119960,0.143766
2019-06-28,0.090156,0.086676,0.110158,0.044153,0.080417,0.085721,0.085299,0.083173,0.083190,0.113564,0.137493
2019-07-05,0.090254,0.086346,0.109490,0.044149,0.080115,0.086605,0.085368,0.083138,0.083218,0.112796,0.138521
2019-07-12,0.090167,0.086240,0.109362,0.043862,0.080048,0.086378,0.085583,0.083180,0.082956,0.112877,0.139347
2019-07-19,0.055557,0.089288,0.113671,0.045443,0.083391,0.089783,0.088796,0.086389,0.086263,0.116510,0.144909


In [6]:
# ============================================================
# BLOCK 5.6 — ALIGN ALL WEEKLY TARGET WEIGHT SETS
# ============================================================

state_targets = (
    state_target_weights
    .reindex(index=timing.index, columns=SECTOR_TICKERS)
)

strategic_targets = strategic_canonical.copy()

for name, w in {
    "Strategic": strategic_targets,
    "TSMOM": tsmom_target_weights,
    "State-dependent": state_targets,
}.items():
    if w.isna().any().any():
        raise ValueError(f"{name} target weights contain missing values.")
    if not (w >= 0).all().all():
        raise ValueError(f"{name} target weights contain negative values.")
    if not np.allclose(w.sum(axis=1), 1.0, atol=1e-10):
        raise ValueError(f"{name} target weights do not sum to 1.")

print("All target-weight sets aligned to the same canonical signal weeks.")


All target-weight sets aligned to the same canonical signal weeks.


## Portfolio-return convention

For each signal week, the associated target weights become effective on the mapped execution date.

The return for a target held from execution date \(E_t\) until the next execution date \(E_{t+1}\) is computed from adjusted closes:

\[
R_{i,t} = \frac{P_{i,E_{t+1}}}{P_{i,E_t}} - 1
\]

Portfolio return is then:

\[
R_{p,t}=\sum_i w_{i,t}R_{i,t}
\]

This is an **execution-close to next-execution-close** weekly holding-period convention. It is deliberately simple, consistent across strategies, and free of same-Friday look-ahead.


### End-of-sample treatment

A target can be known and executed before its **next** execution date has occurred. Such a target is retained in the target-weight datasets, but it is excluded from realized performance until a complete execution-to-execution holding period exists. This avoids both look-ahead and arbitrary partial-week marking.


In [7]:
# ============================================================
# BLOCK 5.7 — COMPLETED EXECUTION-DATE HOLDING-PERIOD RETURNS
# ============================================================

# A realized holding-period return requires TWO observed execution dates:
#
#   target formed at signal t
#   -> executed on E_t
#   -> held until the next observed execution E_{t+1}
#
# The final canonical target can legitimately have an observed execution
# date but no subsequent execution date yet. In that case its forward
# holding-period return is not observable and must NOT be fabricated from
# a partial week or future data.
#
# Therefore:
#   - target weights remain available for every canonical signal week;
#   - realized portfolio returns use only completed execution-to-execution
#     intervals.

execution_dates = pd.DatetimeIndex(
    pd.to_datetime(timing["execution_date"].values)
)

if len(execution_dates) < 2:
    raise RuntimeError(
        "At least two observed execution dates are required "
        "to construct a completed holding-period return."
    )

REALIZED_SIGNAL_WEEKS = timing.index[:-1]
holding_start_dates = execution_dates[:-1]
holding_end_dates = execution_dates[1:]

assert len(REALIZED_SIGNAL_WEEKS) == len(holding_start_dates)
assert len(holding_start_dates) == len(holding_end_dates)
assert (holding_end_dates > holding_start_dates).all()

asset_holding_returns = pd.DataFrame(
    index=REALIZED_SIGNAL_WEEKS,
    columns=SECTOR_TICKERS,
    dtype=float,
)

spy_holding_returns = pd.Series(
    index=REALIZED_SIGNAL_WEEKS,
    dtype=float,
    name="SPY",
)

for signal_week, start_date, end_date in zip(
    REALIZED_SIGNAL_WEEKS,
    holding_start_dates,
    holding_end_dates,
):
    if start_date not in daily_close.index:
        raise KeyError(
            f"Execution start date not in daily prices: {start_date}"
        )

    if end_date not in daily_close.index:
        raise KeyError(
            f"Execution end date not in daily prices: {end_date}"
        )

    start_px = daily_close.loc[start_date, SECTOR_TICKERS]
    end_px = daily_close.loc[end_date, SECTOR_TICKERS]

    asset_holding_returns.loc[signal_week] = (
        end_px / start_px - 1.0
    ).values

    spy_holding_returns.loc[signal_week] = (
        daily_close.loc[end_date, BENCHMARK_TICKER]
        / daily_close.loc[start_date, BENCHMARK_TICKER]
        - 1.0
    )

assert asset_holding_returns.notna().all().all()
assert spy_holding_returns.notna().all()

UNREALIZED_FINAL_SIGNAL_WEEK = timing.index[-1]
UNREALIZED_FINAL_EXECUTION_DATE = execution_dates[-1]

print("Completed holding-period returns constructed.")
print(
    "First completed holding period:",
    holding_start_dates[0].date(),
    "to",
    holding_end_dates[0].date(),
)
print(
    "Last completed holding period:",
    holding_start_dates[-1].date(),
    "to",
    holding_end_dates[-1].date(),
)
print(
    "Latest target retained but excluded from realized returns:",
    UNREALIZED_FINAL_SIGNAL_WEEK.date(),
    "-> execution",
    UNREALIZED_FINAL_EXECUTION_DATE.date(),
)


Completed holding-period returns constructed.
First completed holding period: 2019-06-24 to 2019-07-01
Last completed holding period: 2026-08-17 to 2026-08-24
Latest target retained but excluded from realized returns: 2026-08-21 -> execution 2026-08-24


In [8]:
# ============================================================
# BLOCK 5.8 — PORTFOLIO RETURNS
# ============================================================

def portfolio_returns_from_weights(
    weights: pd.DataFrame,
    asset_returns: pd.DataFrame,
) -> pd.Series:
    aligned_weights = weights.reindex(
        index=asset_returns.index,
        columns=asset_returns.columns,
    )

    if aligned_weights.isna().any().any():
        raise ValueError(
            "Portfolio weights missing after realized-return alignment."
        )

    return (aligned_weights * asset_returns).sum(axis=1)


strategic_returns = portfolio_returns_from_weights(
    strategic_targets,
    asset_holding_returns,
).rename("Strategic_IV")

tsmom_returns_portfolio = portfolio_returns_from_weights(
    tsmom_target_weights,
    asset_holding_returns,
).rename("TSMOM_12M")

state_returns = portfolio_returns_from_weights(
    state_targets,
    asset_holding_returns,
).rename("State_Dependent")

portfolio_returns = pd.concat(
    [
        spy_holding_returns.rename("SPY"),
        strategic_returns,
        tsmom_returns_portfolio,
        state_returns,
    ],
    axis=1,
)

assert portfolio_returns.index.equals(REALIZED_SIGNAL_WEEKS)
assert portfolio_returns.notna().all().all()

display(portfolio_returns.head())


,SPY,Strategic_IV,TSMOM_12M,State_Dependent
Date,,,,
2019-06-21,0.006879,0.000337,-0.000092,0.000337
2019-06-28,0.003923,0.007986,0.008493,0.007986
2019-07-05,0.013240,0.009031,0.008980,0.009031
2019-07-12,-0.009476,-0.011614,-0.011583,-0.011614
2019-07-19,0.011950,0.010169,0.010034,0.010169


In [9]:
# ============================================================
# BLOCK 5.9 — TURNOVER DIAGNOSTICS
# ============================================================

def one_way_turnover(weights: pd.DataFrame) -> pd.Series:
    return (
        0.5
        * weights.diff().abs().sum(axis=1)
    ).fillna(0.0)


turnover = pd.DataFrame(
    {
        "Strategic_IV": one_way_turnover(strategic_targets),
        "TSMOM_12M": one_way_turnover(tsmom_target_weights),
        "State_Dependent": one_way_turnover(state_targets),
    }
)

turnover_summary = pd.DataFrame(
    {
        "mean_weekly_turnover": turnover.mean(),
        "median_weekly_turnover": turnover.median(),
        "max_weekly_turnover": turnover.max(),
        "annualized_turnover_approx": turnover.mean() * 52,
    }
)

display(turnover_summary)


,mean_weekly_turnover,median_weekly_turnover,max_weekly_turnover,annualized_turnover_approx
Strategic_IV,0.005964,0.004450,0.044502,0.310154
TSMOM_12M,0.026070,0.011320,0.139095,1.355629
State_Dependent,0.011788,0.010025,0.084673,0.612997


In [10]:
# ============================================================
# BLOCK 5.10 — CUMULATIVE WEALTH INDEX
# ============================================================

wealth_index = (1.0 + portfolio_returns).cumprod()

display(wealth_index.tail())

print("Terminal wealth index:")
display(
    wealth_index.iloc[-1]
    .rename("terminal_wealth")
    .to_frame()
)


,SPY,Strategic_IV,TSMOM_12M,State_Dependent
Date,,,,
2026-07-17,2.787707,2.288163,2.299061,2.492267
2026-07-24,2.857787,2.300848,2.307164,2.501154
2026-07-31,2.915722,2.325645,2.332029,2.530228
2026-08-07,2.914364,2.331119,2.337518,2.535223
2026-08-14,2.879664,2.343816,2.350249,2.559913


Terminal wealth index:


,terminal_wealth
SPY,2.879664
Strategic_IV,2.343816
TSMOM_12M,2.350249
State_Dependent,2.559913


In [11]:
# ============================================================
# BLOCK 5.11 — SANITY CHECKS
# ============================================================

# Realized return series all share the same completed holding periods.
assert portfolio_returns.index.equals(asset_holding_returns.index)
assert portfolio_returns.index.equals(REALIZED_SIGNAL_WEEKS)

# Every strategy uses exactly the same sector holding-period returns.
for w in [
    strategic_targets,
    tsmom_target_weights,
    state_targets,
]:
    realized_w = w.reindex(REALIZED_SIGNAL_WEEKS)

    assert realized_w.index.equals(asset_holding_returns.index)
    assert realized_w.notna().all().all()
    assert (realized_w >= 0).all().all()
    assert np.allclose(
        realized_w.sum(axis=1),
        1.0,
        atol=1e-10,
    )

# Full target-weight histories also remain long-only and fully invested,
# including the latest target whose forward return is not yet observable.
for w in [
    strategic_targets,
    tsmom_target_weights,
    state_targets,
]:
    assert (w >= 0).all().all()
    assert np.allclose(
        w.sum(axis=1),
        1.0,
        atol=1e-10,
    )

# TSMOM raw multipliers are exactly 0.75 or 1.25.
tsmom_raw_multiplier = 1.0 + tsmom_active
assert set(
    np.round(tsmom_raw_multiplier.stack().unique(), 10)
).issubset({0.75, 1.25})

# Every realized holding period starts after its corresponding signal.
realized_signal_dates = pd.to_datetime(
    timing.loc[REALIZED_SIGNAL_WEEKS, "signal_observation_date"]
)

assert (
    pd.Series(
        holding_start_dates,
        index=REALIZED_SIGNAL_WEEKS,
    )
    > realized_signal_dates
).all()

# The final target is intentionally retained without inventing its
# not-yet-observable next holding-period return.
assert UNREALIZED_FINAL_SIGNAL_WEEK == timing.index[-1]
assert (
    UNREALIZED_FINAL_SIGNAL_WEEK
    not in portfolio_returns.index
)

print("All Block 5 architecture and timing checks passed.")


All Block 5 architecture and timing checks passed.


In [12]:
# ============================================================
# BLOCK 5.12 — SAVE BLOCK 5 DATASETS
# ============================================================

TSMOM_ACTIVE_PATH = (
    DIRS["data_processed"]
    / "weekly_tsmom_active_multipliers.parquet"
)

TSMOM_TARGET_PATH = (
    DIRS["data_processed"]
    / "weekly_tsmom_target_weights.parquet"
)

HOLDING_RETURNS_PATH = (
    DIRS["data_processed"]
    / "weekly_execution_holding_returns_sectors.parquet"
)

PORTFOLIO_RETURNS_PATH = (
    DIRS["data_processed"]
    / "weekly_portfolio_returns_core_strategies.parquet"
)

TURNOVER_PATH = (
    DIRS["data_processed"]
    / "weekly_portfolio_turnover_core_strategies.parquet"
)

WEALTH_PATH = (
    DIRS["data_processed"]
    / "weekly_wealth_index_core_strategies.parquet"
)

TURNOVER_SUMMARY_PATH = (
    DIRS["tables"]
    / "block_5_turnover_summary.csv"
)

tsmom_active.to_parquet(TSMOM_ACTIVE_PATH)
tsmom_target_weights.to_parquet(TSMOM_TARGET_PATH)
asset_holding_returns.to_parquet(HOLDING_RETURNS_PATH)
portfolio_returns.to_parquet(PORTFOLIO_RETURNS_PATH)
turnover.to_parquet(TURNOVER_PATH)
wealth_index.to_parquet(WEALTH_PATH)
turnover_summary.to_csv(TURNOVER_SUMMARY_PATH)

print("Saved:")
for p in [
    TSMOM_ACTIVE_PATH,
    TSMOM_TARGET_PATH,
    HOLDING_RETURNS_PATH,
    PORTFOLIO_RETURNS_PATH,
    TURNOVER_PATH,
    WEALTH_PATH,
    TURNOVER_SUMMARY_PATH,
]:
    print(" ", p)


Saved:
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_tsmom_active_multipliers.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_tsmom_target_weights.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_execution_holding_returns_sectors.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_portfolio_returns_core_strategies.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model

In [13]:
# ============================================================
# BLOCK 5.13 — SAVE BLOCK 5 MANIFEST
# ============================================================

block5_manifest = {
    "project": block1["project"],
    "block": (
        "Block 5 - Benchmark Active Weights & Portfolio Engine"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "strategic_weight_method": block2["strategic_weight_method"],

    "tsmom_specification": {
        "lookback_weeks": 52,
        "positive_signal_active_multiplier": TSMOM_ACTIVE_MAGNITUDE,
        "negative_signal_active_multiplier": -TSMOM_ACTIVE_MAGNITUDE,
        "raw_multiplier_positive": 1.0 + TSMOM_ACTIVE_MAGNITUDE,
        "raw_multiplier_negative": 1.0 - TSMOM_ACTIVE_MAGNITUDE,
    },

    "execution_convention": {
        "signal_frequency": CONFIG["signal_frequency"],
        "execution_rule": CONFIG["rebalance_execution_rule"],
        "holding_period_return_method": (
            "Adjusted close on execution date to adjusted close on next execution date"
        ),
        "same_holding_periods_for_all_strategies": True,
    },

    "portfolios": [
        "SPY",
        "Strategic_IV",
        "TSMOM_12M",
        "State_Dependent",
    ],

    "canonical_signal_start": str(timing.index.min().date()),
    "canonical_signal_end": str(timing.index.max().date()),
    "canonical_execution_start": str(holding_start_dates[0].date()),
    "canonical_execution_end": str(execution_dates[-1].date()),
    "realized_return_signal_start": str(REALIZED_SIGNAL_WEEKS.min().date()),
    "realized_return_signal_end": str(REALIZED_SIGNAL_WEEKS.max().date()),
    "realized_return_execution_start": str(holding_start_dates[0].date()),
    "realized_return_execution_end": str(holding_end_dates[-1].date()),
    "latest_target_signal_week_without_forward_return": str(
        UNREALIZED_FINAL_SIGNAL_WEEK.date()
    ),
    "latest_target_execution_date_without_forward_return": str(
        UNREALIZED_FINAL_EXECUTION_DATE.date()
    ),

    "lookahead_controls": [
        (
            "TSMOM signal is calculated on completed Friday-labelled data."
        ),
        (
            "Target weights become effective only on the mapped next U.S. trading session."
        ),
        (
            "Portfolio returns are measured from execution close to next execution close."
        ),
        (
            "The latest target is retained even when its next execution date "
            "has not yet occurred; no partial or future holding-period return "
            "is fabricated."
        ),
        (
            "All active strategies use identical holding-period sector returns."
        ),
    ],

    "saved_files": {
        "tsmom_active_multipliers": str(TSMOM_ACTIVE_PATH),
        "tsmom_target_weights": str(TSMOM_TARGET_PATH),
        "sector_holding_returns": str(HOLDING_RETURNS_PATH),
        "portfolio_returns": str(PORTFOLIO_RETURNS_PATH),
        "portfolio_turnover": str(TURNOVER_PATH),
        "wealth_index": str(WEALTH_PATH),
        "turnover_summary": str(TURNOVER_SUMMARY_PATH),
    },
}

BLOCK5_MANIFEST = (
    DIRS["manifests"] / "block_5_portfolio_engine.json"
)

with open(BLOCK5_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(block5_manifest, f, indent=2)

print("Saved Block 5 manifest:")
print(BLOCK5_MANIFEST)


Saved Block 5 manifest:
/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/manifests/block_5_portfolio_engine.json


In [14]:
# ============================================================
# BLOCK 5.14 — FINAL STATUS
# ============================================================

summary = pd.Series(
    {
        "Sector count": len(SECTOR_TICKERS),
        "Benchmark": BENCHMARK_TICKER,
        "Strategic allocation": block2["strategic_weight_method"],
        "TSMOM lookback weeks": 52,
        "TSMOM active tilt": "+/- 0.25",
        "Signal frequency": CONFIG["signal_frequency"],
        "Execution rule": CONFIG["rebalance_execution_rule"],
        "Canonical signal start": timing.index.min().date(),
        "Canonical signal end": timing.index.max().date(),
        "First realized execution": holding_start_dates[0].date(),
        "Last realized execution": holding_end_dates[-1].date(),
        "Latest target execution": UNREALIZED_FINAL_EXECUTION_DATE.date(),
        "Canonical target weeks": len(timing),
        "Completed return observations": len(portfolio_returns),
        "Latest target awaiting forward return": True,
        "Return series complete": bool(
            portfolio_returns.notna().all().all()
        ),
        "Strategic weights sum to 1": bool(
            np.allclose(strategic_targets.sum(axis=1), 1.0)
        ),
        "TSMOM weights sum to 1": bool(
            np.allclose(tsmom_target_weights.sum(axis=1), 1.0)
        ),
        "State-dependent weights sum to 1": bool(
            np.allclose(state_targets.sum(axis=1), 1.0)
        ),
        "All portfolios long-only": bool(
            (strategic_targets >= 0).all().all()
            and (tsmom_target_weights >= 0).all().all()
            and (state_targets >= 0).all().all()
        ),
    },
    name="Block 5 status",
).to_frame()

display(summary)

print("\nBLOCK 5 COMPLETE")
print("Next: Block 6 — Performance Analytics & Attribution")


,Block 5 status
Sector count,11
Benchmark,SPY
Strategic allocation,INVERSE_VOLATILITY_52W
TSMOM lookback weeks,52
TSMOM active tilt,+/- 0.25
Signal frequency,W-FRI
Execution rule,NEXT_US_TRADING_SESSION_AFTER_SIGNAL
Canonical signal start,2019-06-21
Canonical signal end,2026-08-21
First realized execution,2019-06-24



BLOCK 5 COMPLETE
Next: Block 6 — Performance Analytics & Attribution
